# ColQwen2 GPU Embedding — InsureRAG-VLM

Run this notebook on a GPU instance (Google Colab Pro A100, Lambda Labs, or Vast.ai).

**What this does:**
1. Clones the InsureRAG-VLM repo
2. Installs GPU dependencies
3. Generates ColQwen2 page-image embeddings for the real insurance PDFs
4. Runs retrieval metrics
5. Downloads a compact metrics JSON to commit to the repo

**GPU requirement:** ~8 GB VRAM (A100/T4/V100 all work)

**Time:** ~15-30 minutes for 64 pages

In [ ]:
# Check GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — stop here and switch to a GPU runtime')

In [ ]:
# Clone repo (replace with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/InsureRAG-VLM.git
%cd InsureRAG-VLM

In [ ]:
# Install CPU + GPU dependencies
!pip install -r requirements.txt -q
!pip install -r requirements-gpu.txt -q
print('Dependencies installed')

In [ ]:
# Download real insurance PDFs from state department sources
!python main.py import-data --datasets public_docs
!ls data/00_raw/external/public_docs/

In [ ]:
# Preprocess: render PDFs to page images + generate page manifest
!python main.py preprocess-pages data/00_raw/external/public_docs --output-root data --render-dpi 150
!wc -l data/03_index/colqwen2/page_manifest.jsonl

In [ ]:
# Build ColQwen2 visual index (downloads ~5 GB model on first run)
# This encodes each page image as a 128-dim ColQwen2 embedding
!python main.py build-visual-index data/03_index/colqwen2/page_manifest.jsonl \
    --backend colqwen2_local \
    --index-dir data/03_index/colqwen2

import os
embed_path = 'data/03_index/colqwen2/colqwen2_local.pt'
size_mb = os.path.getsize(embed_path) / 1e6 if os.path.exists(embed_path) else 0
print(f'Embedding file: {size_mb:.1f} MB')

In [ ]:
# Generate QA pairs from real PDFs (300 answerable + unsupported)
!python main.py generate-qa data/00_raw/external/public_docs \
    --output-dir data/02_processed \
    --target-count 300
!wc -l data/02_processed/qa_pairs.jsonl

In [ ]:
# Run ColQwen2 visual retrieval metrics
!python main.py visual-retrieval-metrics data/02_processed/qa_pairs.jsonl \
    --backend colqwen2_local \
    --index-dir data/03_index/colqwen2 \
    --top-k 5

In [ ]:
# Run text baseline for comparison
!python main.py build-index data/00_raw/external/public_docs
!python main.py retrieval-metrics data/00_raw/external/public_docs data/02_processed/qa_pairs.jsonl --top-k 5

In [ ]:
# Run full ablation and save report
!mkdir -p reports/ablation_gpu
!python main.py run-ablation \
    --data-folder data/00_raw/external/public_docs \
    --qa-path data/02_processed/qa_pairs.jsonl \
    --output-dir reports/ablation_gpu \
    --visual-index-dir data/03_index/colqwen2

# Show summary
with open('reports/ablation_gpu/summary.md') as f:
    print(f.read())

In [ ]:
# Save compact metrics JSON (commit this, NOT the raw embeddings)
import json, csv
from pathlib import Path

metrics = {}
with open('reports/ablation_gpu/retrieval_metrics.csv') as f:
    for row in csv.DictReader(f):
        backend = row.pop('backend')
        metrics[backend] = {k: float(v) for k, v in row.items() if v}

out = Path('reports/visual_retrieval_colqwen2.json')
out.write_text(json.dumps({'n_qa': 300, 'backends': metrics}, indent=2))
print(out.read_text())

In [ ]:
# Download metrics file to commit to git
# (Do NOT commit the .pt embedding file — too large)
from google.colab import files
files.download('reports/visual_retrieval_colqwen2.json')
files.download('reports/ablation_gpu/summary.md')
files.download('reports/ablation_gpu/retrieval_metrics.csv')